# Business Problem Analysis — SampleSuperstore

## Tujuan
Notebook ini digunakan untuk menemukan **business problem berdasarkan dataset**, bukan berdasarkan asumsi kasus sebelumnya.

Alur analisis:
**Dataset → Data Understanding → KPI → Problem Identification → Potential Causes → Data Evidence → Insights**

> **Catatan:** Business problem dan potential causes di bawah harus didasarkan pada pola yang ditemukan dari data. Potential cause disebut sebagai *hypothesis* sampai hubungan sebab-akibat benar-benar dapat dibuktikan.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

df = pd.read_csv('SampleSuperstore.csv')
df.head()

## 1. Data Understanding

In [ ]:
print("Jumlah baris:", df.shape[0])
print("Jumlah kolom:", df.shape[1])
print("\nKolom:")
print(df.columns.tolist())

print("\nMissing values:")
display(df.isna().sum().to_frame('Missing Values'))

print("\nData types:")
display(df.dtypes.to_frame('Data Type'))

## 2. KPI Utama

KPI yang digunakan:
- **Total Sales**
- **Total Profit**
- **Profit Margin**
- **Average Discount**
- **Total Quantity**

Profit Margin digunakan untuk melihat apakah tingginya penjualan juga menghasilkan profit yang sehat.

In [ ]:
total_sales = df['Sales'].sum()
total_profit = df['Profit'].sum()
profit_margin = total_profit / total_sales * 100
avg_discount = df['Discount'].mean() * 100
total_quantity = df['Quantity'].sum()

kpi = pd.DataFrame({
    'KPI': ['Total Sales', 'Total Profit', 'Profit Margin', 'Average Discount', 'Total Quantity'],
    'Value': [
        total_sales,
        total_profit,
        f'{profit_margin:.2f}%',
        f'{avg_discount:.2f}%',
        total_quantity
    ]
})
display(kpi)

## 3. Analisis Profitabilitas per Category

In [ ]:
category = df.groupby('Category').agg(
    Sales=('Sales','sum'),
    Profit=('Profit','sum'),
    Quantity=('Quantity','sum'),
    Transactions=('Category','size'),
    Avg_Discount=('Discount','mean')
).reset_index()

category['Profit_Margin'] = category['Profit'] / category['Sales'] * 100
category = category.sort_values('Profit_Margin')
display(category)

category.plot(x='Category', y='Profit_Margin', kind='bar', legend=False)
plt.title('Profit Margin by Category')
plt.ylabel('Profit Margin (%)')
plt.xlabel('Category')
plt.xticks(rotation=0)
plt.show()

### Problem 1 — Furniture memiliki profitabilitas sangat rendah

Furniture menghasilkan sales yang besar, tetapi profit margin-nya jauh lebih rendah dibanding kategori lainnya.

**Business Problem:**
> Perusahaan memiliki profitabilitas yang rendah pada kategori Furniture meskipun kategori tersebut menghasilkan sales yang besar.

**Potential causes / hypotheses:**
1. Tingkat discount Furniture relatif tinggi.
2. Terdapat sub-category Furniture yang mengalami profit negatif.

Kedua penyebab tersebut perlu divalidasi melalui drill-down.

In [ ]:
furniture = df[df['Category'] == 'Furniture']

furniture_summary = pd.DataFrame({
    'Metric': ['Sales', 'Profit', 'Profit Margin', 'Average Discount'],
    'Value': [
        furniture['Sales'].sum(),
        furniture['Profit'].sum(),
        f'{furniture["Profit"].sum()/furniture["Sales"].sum()*100:.2f}%',
        f'{furniture["Discount"].mean()*100:.2f}%'
    ]
})
display(furniture_summary)

## 4. Drill-down Furniture ke Sub-Category

In [ ]:
furniture_sub = furniture.groupby('Sub-Category').agg(
    Sales=('Sales','sum'),
    Profit=('Profit','sum'),
    Quantity=('Quantity','sum'),
    Avg_Discount=('Discount','mean')
).reset_index()

furniture_sub['Profit_Margin'] = furniture_sub['Profit'] / furniture_sub['Sales'] * 100
display(furniture_sub.sort_values('Profit_Margin'))

### Problem 2 — Tables mengalami kerugian

Sub-category **Tables** memiliki sales yang cukup besar tetapi mencatat profit negatif.

**Business Problem:**
> Sub-category Tables menghasilkan penjualan tetapi tidak menghasilkan profit, sehingga berpotensi mengurangi profitabilitas perusahaan.

**Potential causes / hypotheses:**
- Average discount Tables relatif tinggi.
- Struktur biaya/margin produk Tables mungkin tidak mampu menutup dampak discount.

> Dataset tidak memiliki kolom Cost, sehingga struktur biaya tidak dapat dibuktikan secara langsung.

In [ ]:
tables = df[df['Sub-Category'] == 'Tables']

tables_summary = pd.DataFrame({
    'Metric': ['Sales', 'Profit', 'Profit Margin', 'Average Discount', 'Quantity'],
    'Value': [
        tables['Sales'].sum(),
        tables['Profit'].sum(),
        f'{tables["Profit"].sum()/tables["Sales"].sum()*100:.2f}%',
        f'{tables["Discount"].mean()*100:.2f}%',
        tables['Quantity'].sum()
    ]
})
display(tables_summary)

## 5. Analisis Profitabilitas per Region

In [ ]:
region = df.groupby('Region').agg(
    Sales=('Sales','sum'),
    Profit=('Profit','sum'),
    Transactions=('Region','size'),
    Avg_Discount=('Discount','mean')
).reset_index()

region['Profit_Margin'] = region['Profit'] / region['Sales'] * 100
display(region.sort_values('Profit_Margin'))

region.plot(x='Region', y='Profit_Margin', kind='bar', legend=False)
plt.title('Profit Margin by Region')
plt.ylabel('Profit Margin (%)')
plt.xlabel('Region')
plt.xticks(rotation=0)
plt.show()

### Problem 3 — Region Central memiliki profit margin terendah

Central memiliki profit margin paling rendah dibandingkan seluruh region.

**Business Problem:**
> Region Central memiliki profitabilitas yang relatif rendah dibandingkan region lainnya.

**Potential causes / hypotheses:**
1. Average discount di Central paling tinggi.
2. Komposisi category/sub-category di Central mungkin lebih banyak berasal dari produk ber-margin rendah.

Komposisi produk perlu dianalisis lebih lanjut sebelum menyatakan penyebab secara definitif.

In [ ]:
central_category = df[df['Region'] == 'Central'].groupby('Category').agg(
    Sales=('Sales','sum'),
    Profit=('Profit','sum'),
    Avg_Discount=('Discount','mean')
).reset_index()

central_category['Profit_Margin'] = central_category['Profit'] / central_category['Sales'] * 100
display(central_category.sort_values('Profit_Margin'))

## 6. Lima Data Evidence Utama

In [ ]:
evidence = pd.DataFrame({
    'No': [1, 2, 3, 4, 5],
    'Data Evidence': [
        'Furniture Profit Margin',
        'Tables Profit',
        'Tables Average Discount',
        'Central Profit Margin',
        'Central Average Discount'
    ],
    'Value': [
        f'{furniture["Profit"].sum()/furniture["Sales"].sum()*100:.2f}%',
        f'${tables["Profit"].sum():,.2f}',
        f'{tables["Discount"].mean()*100:.2f}%',
        f'{region.loc[region["Region"]=="Central", "Profit_Margin"].iloc[0]:.2f}%',
        f'{region.loc[region["Region"]=="Central", "Avg_Discount"].iloc[0]*100:.2f}%'
    ]
})
display(evidence)

## 7. Tiga Key Insights

1. **Sales besar tidak selalu berarti profit tinggi.** Furniture memiliki sales sekitar $742 ribu tetapi profit margin hanya sekitar 2,49%.
2. **Tables merupakan area produk yang perlu diprioritaskan.** Sub-category ini mencatat profit negatif sekitar $17,73 ribu dan average discount sekitar 26,13%.
3. **Central membutuhkan perhatian dari sisi profitabilitas.** Region ini memiliki profit margin sekitar 7,92%, terendah di antara empat region, dengan average discount sekitar 24,04%.

## 8. Problem Tree

```text
BUSINESS PROBLEM
│
├── Problem 1: Profitabilitas Furniture rendah
│   ├── Potential Cause A: Discount relatif tinggi
│   └── Potential Cause B: Ada sub-category dengan profit rendah/negatif
│
├── Problem 2: Tables mengalami kerugian
│   ├── Potential Cause A: Discount tinggi
│   └── Potential Cause B: Margin produk tidak cukup menutup dampak discount
│
└── Problem 3: Profitabilitas Central rendah
    ├── Potential Cause A: Discount tertinggi antar-region
    └── Potential Cause B: Komposisi produk ber-margin rendah perlu ditinjau
```

## 9. Kesimpulan

Berdasarkan analisis dataset SampleSuperstore, ditemukan tiga business problem utama: **profitabilitas Furniture yang rendah, kerugian pada Tables, dan profitabilitas Region Central yang paling rendah**.

Analisis ini menggunakan evidence yang tersedia pada dataset. Potential causes masih merupakan **hipotesis** dan tidak boleh dianggap sebagai hubungan sebab-akibat yang sudah terbukti.

Dataset juga tidak memiliki kolom tanggal, sehingga analisis tren penjualan berdasarkan periode waktu tidak dilakukan.